# Capítulo 6 — RAG e agentes

## Missão
Dar ao assistente o conhecimento da Nimbus com **RAG**. Você vai:
- quebrar os documentos em pedaços;
- buscar por palavras (BM25) e por sentido (embeddings);
- combinar as duas buscas (RRF) e medir a recuperação (recall@k, MRR);
- responder com citações.

Depois, um **agente** que usa ferramentas (calcular preço, consultar a documentação) num laço de raciocínio, e uma **memória** de conversa. No fim, você compara com a linha de base do capítulo 4 e vê o salto.

**No fim deste caderno você terá:** os módulos `meu_assistente/rag.py` (base de conhecimento com busca híbrida e resposta com fontes) e `meu_assistente/agente.py` (agente com ferramentas).

**Leitura essencial (opcional, livro em inglês):**
- RAG Architecture · PDF p. 280
- Retrieval Algorithms · PDF p. 281
- Retrieval Optimization · PDF p. 291
- Agent Overview e Tools · PDF p. 300
- Planning · PDF p. 305
- Agent Failure Modes and Evaluation · PDF p. 322
- Memory · PDF p. 324

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap06_rag_e_agentes.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_assistente/
os.makedirs("../meu_assistente", exist_ok=True)
open("../meu_assistente/__init__.py", "a").close()        # meu_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: meu_assistente")

## Ideia em 30 segundos
**RAG** = buscar os trechos certos e colocá-los no prompt. A qualidade do RAG depende mais da **busca** que do modelo:
- BM25 é ótimo para termos exatos ("NIM-123", "SSO");
- embeddings acham o sentido mesmo com outras palavras;
- a busca híbrida junta os dois.

Um **agente** é um modelo num laço: ele planeja, chama ferramentas, lê o resultado e decide o próximo passo, até responder.

### 6.1 **Pedaços (chunks).** Quebre cada documento em pedaços de **2 frases**, com sobreposição de 1 frase (a última frase de um pedaço é a primeira do próximo). Guarde dicts com `doc`, `titulo` e `texto`. Quantos pedaços saíram?

### 6.2 **Busca por palavras (BM25).** Use `rank_bm25.BM25Okapi` com os textos tokenizados (minúsculas, só palavras). Escreva `buscar_bm25(pergunta, k=3)`, que devolve os índices dos k melhores pedaços, e teste com "Como ligo commits às tarefas?".

### 6.3 **Busca por sentido (embeddings).** Calcule os vetores de todos os pedaços **uma vez** (`VETORES`). Escreva `buscar_vetor(pergunta, k=3)` com similaridade de cosseno e teste com uma pergunta que não usa as palavras do documento: "Dá para mexer nas tarefas no avião?".

### 6.4 **Meça a recuperação.** Para cada item de `PERGUNTAS`, veja se o `doc` certo aparece nos 3 primeiros pedaços (**recall@3**) e em que posição aparece pela primeira vez (**MRR** = média de 1/posição). Compare BM25 e embeddings.

### 6.5 **Busca híbrida com RRF.** Junte as duas listas com *Reciprocal Rank Fusion*: cada pedaço ganha `1 / (60 + posição)` em cada lista em que aparece (posição começando em 1). Busque 10 em cada método, some e devolva os k melhores. Meça de novo.

### 6.6 **Reescrita da consulta.** Clientes escrevem de jeito informal. Peça ao modelo para reescrever "to travado nao consigo logar ajuda" como uma consulta de busca objetiva e compare os pedaços recuperados antes e depois.

### 6.7 **Módulo de RAG.** Escreva `rag.py` com a classe `BaseConhecimento(documentos)`:
- `__init__`: cria os pedaços, o BM25 e os vetores;
- `buscar(pergunta, k=3)`: busca híbrida com RRF, devolve a lista de pedaços;
- `responder(pergunta, modelo=MODELO)`: monta o prompt com os pedaços numerados `[1]`, `[2]`..., manda o modelo usar **só** esse contexto e citar os números, e devolve o mesmo dict do `nucleo` (texto, latência, tokens, modelo) mais `fontes` (ids dos documentos).

_A célula grava `meu_assistente/rag.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/rag.py


### 6.8 **O salto do RAG.** Crie `base = BaseConhecimento(DOCUMENTOS)`, responda 2 perguntas mostrando as fontes e depois avalie `base.responder` nas mesmas 8 perguntas do capítulo 4 com `meu_assistente.avaliacao`. Compare com `../saidas/base_cap04.csv`.

## Agentes
O modelo recebe a descrição das **ferramentas** (nome, para que serve, parâmetros em JSON Schema). Quando precisa, ele devolve um pedido de chamada (`tool_calls`) em vez de texto. O seu código executa a ferramenta, devolve o resultado numa mensagem `role="tool"` e chama o modelo de novo, até ele responder.

Os erros mais comuns de um agente são:
- ferramenta errada;
- argumentos inválidos;
- laço infinito.

Por isso sempre há um **limite de passos** e um registro (*trace*) de tudo.

### 6.9 **Ferramentas.** Escreva duas funções Python:
- `calcular_preco(plano, usuarios, anual)`, com preços por usuário por mês: Equipe 29 e Empresa 59; o pagamento anual tem 20% de desconto; o Grátis custa 0;
- `buscar_documentacao(consulta)`, que devolve o texto dos 2 pedaços mais relevantes da `base`.

Escreva também a lista `FERRAMENTAS` no formato da API (`type: function`, `name`, `description`, `parameters`).

### 6.10 **Módulo do agente.** Escreva `agente.py` com `executar_agente(pergunta, ferramentas, funcoes, max_passos=5)`:
1. começa com um sistema que manda usar ferramentas quando precisar de dados;
2. a cada passo chama `conversar(..., tools=ferramentas)`;
3. se vierem `tool_calls`, executa cada uma (`funcoes[nome](**args)`), captura erros como texto e devolve com `role="tool"` e `tool_call_id`;
4. se vier texto, termina.

Devolve `{"resposta": ..., "passos": [trace]}`. Se estourar `max_passos`, devolve uma mensagem de desistência.

_A célula grava `meu_assistente/agente.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/agente.py


### 6.11 **Rode o agente.** Pergunte "Quanto uma equipe de 12 pessoas pagaria por ano no plano Empresa pagando anual? E tem SSO?". Mostre a resposta e o *trace* como tabela. Ele usou as duas ferramentas? (No modo simulado as chamadas são falsas e podem dar erro de argumento: veja como o erro vira texto para o modelo.)

### 6.12 **Memória de conversa.** Escreva a classe `MemoriaResumida(max_mensagens=4)`: guarda as mensagens. Quando passar do limite, junta as mais antigas num **resumo** (feito pelo modelo) e mantém só as últimas. O método `contexto()` devolve a lista para mandar ao modelo (o resumo como mensagem de sistema + as recentes). Simule 6 trocas e mostre o contexto final.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero testar com o modelo de verdade (fora do modo simulado):
